# Project Assignment 2: Data Cleaning & Transformation

**Objective:**
In the previous assignment (Lab 5), you created a roadmap and identified the columns necessary for your research. In this assignment, you will execute that plan. You will perform the "dirty work" of data science: handling missing values, fixing data types, and transforming raw data into a format ready for analysis.

**Guidelines:**
* **Focus:** Only clean the data relevant to your research questions.
* **Justification:** You must explain *why* you chose a specific cleaning strategy (e.g., why did you drop a row vs. imputing the value?).
* **Reference:** Use the techniques covered in **Lecture 6** (Handling Missing Data, Transformations).

In [3]:
# --- Import Libraries ---
import pandas as pd
import numpy as np

# TODO: Load your dataset (use the filtered/subsetted data from Lab 5 if available)
df = pd.read_csv('../data/amazon.csv')
(df.isna().sum().sort_values(ascending=False)).head(15)

order_id            0
order_date          0
product_id          0
product_category    0
price               0
discount_percent    0
quantity_sold       0
customer_region     0
payment_method      0
rating              0
review_count        0
discounted_price    0
total_revenue       0
dtype: int64

## Part 1: Handling Missing Data (1.5 Points)

Your first task is to quantify the "missingness" in your dataset and apply a strategy to handle it. Refer to Lecture 6 for methods like `.isna()`, `.dropna()`, and `.fillna()`.

In [7]:
# TODO: Check for missing values in your relevant columns
(df.isna().sum().sort_values(ascending=False)).head(15)

order_id            0
order_date          0
product_id          0
product_category    0
price               0
discount_percent    0
quantity_sold       0
customer_region     0
payment_method      0
rating              0
review_count        0
discounted_price    0
total_revenue       0
dtype: int64

**Analysis of Missing Data:**
*Double click here to edit.*

1.  **Context & Bias:** Look at the specific rows with missing values. If you were to drop these rows, would you be systematically excluding a specific subgroup (e.g., a specific year, region, or demographic)? Explain how this might bias your analysis.

 I checked for missing values across all columns using df.isna().sum() and found that every column has 0 missing values. Since there are no rows with missing data, dropping rows would not remove any specific subgroup (such as a particular region, product category, time period, or payment method). Because nothing is missing, there is no risk of missing-data bias in the analysis.



2.  **Strategy Justification:** For the columns with missing data, explain *why* your chosen strategy (dropping vs. imputing) is the safest approach for your specific research question.
    * *Example:* "I am dropping 'Budget' because imputing an average would hide the distinction between indie and blockbuster films, which is central to my thesis."

    Because the dataset contains no missing values, the safest strategy is to make no changes and keep all rows as-is. Dropping or imputing would be unnecessary and could introduce distortion by removing valid observations or artificially changing true values.

In [5]:
# TODO: Execute your strategy.
# - Option A: Drop rows (df.dropna...)
# - Option B: Impute values (df.fillna...)

# Verify the cleanup:
# df.isna().sum()

missing_total = df.isna().sum().sum()

if missing_total == 0:
    df_clean = df.copy()
else:
    df_clean = df.dropna()

df_clean.isna().sum()

order_id            0
order_date          0
product_id          0
product_category    0
price               0
discount_percent    0
quantity_sold       0
customer_region     0
payment_method      0
rating              0
review_count        0
discounted_price    0
total_revenue       0
dtype: int64

## Part 2: Data Transformations (2.0 Points)

Raw data is rarely in the perfect format. In this section, you will convert data types, remove duplicates, and create new features (columns) that make analysis easier.

### 2.1 Fix Data Types
Ensure dates are actually timestamps and categories are recognized as such.

In [9]:
# TODO: Check current data types
df.dtypes


# TODO: Convert columns if necessary
# Example: df['date_col'] = pd.to_datetime(df['date_col'])
# Example: df['category_col'] = df['category_col'].astype('category')
df2 = df_clean.copy()

df2["order_date"] = pd.to_datetime(df2["order_date"], errors="coerce")

df2 = df2.drop_duplicates()

df2["order_year"] = df2["order_date"].dt.year
df2["order_month"] = df2["order_date"].dt.month
df2["order_day"] = df2["order_date"].dt.day
df2["order_dayofweek"] = df2["order_date"].dt.dayofweek
df2["order_quarter"] = df2["order_date"].dt.quarter

df2["discount_rate"] = df2["discount_percent"] / 100
df2["computed_discounted_price"] = df2["price"] * (1 - df2["discount_rate"])
df2["computed_total_revenue"] = df2["computed_discounted_price"] * df2["quantity_sold"]
df2["is_discounted"] = (df2["discount_percent"] > 0).astype(int)

df2

,order_id,order_date,product_id,product_category,price,discount_percent,quantity_sold,customer_region,payment_method,rating,...,total_revenue,order_year,order_month,order_day,order_dayofweek,order_quarter,discount_rate,computed_discounted_price,computed_total_revenue,is_discounted
0,1,2022-04-13,2637,Books,128.75,10,4,North America,UPI,3.5,...,463.52,2022,4,13,2,2,0.10,115.8750,463.500,1
1,2,2023-03-12,2300,Fashion,302.60,20,5,Asia,Credit Card,3.7,...,1210.40,2023,3,12,6,1,0.20,242.0800,1210.400,1
2,3,2022-09-28,3670,Sports,495.80,20,2,Europe,UPI,4.4,...,793.28,2022,9,28,2,3,0.20,396.6400,793.280,1
3,4,2022-04-17,2522,Books,371.95,15,4,Middle East,UPI,5.0,...,1264.64,2022,4,17,6,2,0.15,316.1575,1264.630,1
4,5,2022-03-13,1717,Beauty,201.68,0,4,Middle East,UPI,4.6,...,806.72,2022,3,13,6,1,0.00,201.6800,806.720,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,49996,2022-09-03,1433,Beauty,26.99,0,5,Middle East,Credit Card,2.4,...,134.95,2022,9,3,5,3,0.00,26.9900,134.950,0
49996,49997,2022-07-03,1428,Beauty,294.23,10,5,Asia,Credit Card,3.1,...,1324.05,2022,7,3,6,3,0.10,264.8070,1324.035,1
49997,49998,2023-02-17,4651,Electronics,352.11,30,4,Asia,Debit Card,3.1,...,985.92,2023,2,17,4,1,0.30,246.4770,985.908,1
49998,49999,2022-09-30,4371,Beauty,307.54,5,1,Middle East,UPI,1.8,...,292.16,2022,9,30,4,3,0.05,292.1630,292.163,1


### 2.2 Deduplication
Ensure you do not have duplicate records skewing your counts.

In [10]:
# TODO: Check for duplicates
# df.duplicated().sum()

# TODO: Drop duplicates if they exist
# df = df.drop_duplicates()
df.duplicated().sum()

df = df.drop_duplicates()

df.duplicated().sum()

np.int64(0)

### 2.3 Feature Engineering (Binning or Categories)
Create meaningful categories from continuous data (e.g., turning "Age" into "Age Group" or "Time" into "Rush Hour/Standard").

In [13]:
# TODO: Create at least one new column based on existing data.
# Ideas:
# - Binning (pd.cut) for price ranges, age groups, or duration.
# - One-Hot Encoding (pd.get_dummies) for categorical variables.
# - Extracting components (e.g., extracting 'Month' from a date column).
df["order_date"] = pd.to_datetime(df["order_date"])
df["order_month"] = df["order_date"].dt.month

print(df.columns)
print(df[["order_date", "order_month"]].head(10))

Index(['order_id', 'order_date', 'product_id', 'product_category', 'price',
       'discount_percent', 'quantity_sold', 'customer_region',
       'payment_method', 'rating', 'review_count', 'discounted_price',
       'total_revenue', 'order_month'],
      dtype='str')
  order_date  order_month
0 2022-04-13            4
1 2023-03-12            3
2 2022-09-28            9
3 2022-04-17            4
4 2022-03-13            3
5 2023-12-02           12
6 2022-01-21            1
7 2023-09-07            9
8 2022-05-02            5
9 2023-04-12            4


## Part 3: Final Quality Check (1.5 Points)

Before you proceed to analysis in the next assignment, you must prove your dataset is clean.

In [14]:
# TODO: Display the .info() summary of your final, clean dataframe.
# It should show:
# 1. Correct data types (no 'objects' for numbers/dates).
# 2. No missing values (unless explicitly justified).
df.info()
df.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   order_id          50000 non-null  int64         
 1   order_date        50000 non-null  datetime64[us]
 2   product_id        50000 non-null  int64         
 3   product_category  50000 non-null  str           
 4   price             50000 non-null  float64       
 5   discount_percent  50000 non-null  int64         
 6   quantity_sold     50000 non-null  int64         
 7   customer_region   50000 non-null  str           
 8   payment_method    50000 non-null  str           
 9   rating            50000 non-null  float64       
 10  review_count      50000 non-null  int64         
 11  discounted_price  50000 non-null  float64       
 12  total_revenue     50000 non-null  float64       
 13  order_month       50000 non-null  int32         
dtypes: datetime64[us](1), float64(4),

order_id            0
order_date          0
product_id          0
product_category    0
price               0
discount_percent    0
quantity_sold       0
customer_region     0
payment_method      0
rating              0
review_count        0
discounted_price    0
total_revenue       0
order_month         0
dtype: int64

**Final Reflection:**
*Double click here to edit.*

1.  **Trade-offs:** Compare your raw row count to your final row count. Explain why the benefits of having "clean" data outweigh the loss of sample size in your specific case.

My raw and final row counts stayed the same because the dataset had no missing values and did not require dropping any rows. The benefit of cleaning still matters because I confirmed there are no duplicates and converted order_date into a proper datetime format, which prevents counting errors and makes time-based analysis accurate. These checks improve the reliability of summaries and trends without reducing the sample size.

2.  **Limitations:** Every data cleaning decision involves compromise. Describe one specific limitation your cleaning strategy has introduced. (e.g., "By removing outliers, I may be ignoring extreme but valid user behaviors.")

One limitation is that converting order_date from a string to a datetime assumes the dates follow a consistent format. If any dates were formatted differently, they could be misread or fail conversion, which would affect the month feature and any time-based results. Another limitation is that if duplicates were removed using order_id, there is a small risk of removing a record that appears duplicated but was actually valid due to how the data was recorded.

In [15]:
# TODO: Save your clean dataset to a new CSV file
df.to_csv('../data/amazon_project_data_clean.csv', index=False)